# Cleanup: Firestore

This notebook cleans up resources created by `setup/05_setup_firestore.ipynb`.

**Resources that will be deleted:**

**1. Firestore Data:**
   - Collection: `datasets` (all documents)
   - Subcollection: `datasets/{dataset}/tables` (all documents)

**2. Firestore Database:**
   - Database: `(default)`

**⚠️  WARNING:**
- This operation cannot be undone
- All Firestore data will be permanently deleted
- The Firestore database will be deleted

**Required permissions:**
- `roles/datastore.owner` (to delete Firestore data and database)

## Section 1: Configuration & Authentication

In [ ]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           

# Firestore
DATABASE_ID = "(default)"
DATASETS_COLLECTION = "datasets"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Firestore Database to delete: {DATABASE_ID}")
print(f"  Collection to delete: {DATASETS_COLLECTION}")

In [ ]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

In [ ]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = ["google-cloud-firestore"]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

In [ ]:
# Initialize clients
from google.cloud import firestore

firestore_client = firestore.Client(project=PROJECT_ID, database=DATABASE_ID)

print("✅ Clients initialized:")
print("  - Firestore client")

## Section 2: Delete Firestore Data

Delete all documents in the `datasets` collection and its subcollections.

**Note:** Firestore documents in subcollections must be deleted before the parent documents.

In [ ]:
# Delete all documents in the Firestore 'datasets' collection and subcollections
print("🗑️  Deleting Firestore data...")
print()

def delete_collection(collection_ref, batch_size=100):
    """Recursively delete all documents in a collection."""
    deleted_count = 0
    docs = list(collection_ref.limit(batch_size).stream())

    while docs:
        for doc in docs:
            # Recursively delete subcollections
            for subcol in doc.reference.collections():
                deleted_count += delete_collection(subcol, batch_size)
            doc.reference.delete()
            deleted_count += 1
        docs = list(collection_ref.limit(batch_size).stream())

    return deleted_count

try:
    datasets_ref = firestore_client.collection(DATASETS_COLLECTION)
    
    # Check if the collection has any documents
    sample = list(datasets_ref.limit(1).stream())
    
    if not sample:
        print(f"⚠️  Collection '{DATASETS_COLLECTION}' is empty or does not exist - skipping")
        firestore_deleted_count = 0
    else:
        print(f"   Deleting all documents in '{DATASETS_COLLECTION}' collection...")
        firestore_deleted_count = delete_collection(datasets_ref)
        print(f"   ✅ Deleted {firestore_deleted_count} document(s) from Firestore")

except Exception as e:
    error_msg = str(e)
    if "not found" in error_msg.lower() or "404" in error_msg:
        print(f"⚠️  Firestore database not found (already deleted or never created)")
        firestore_deleted_count = 0
    else:
        print(f"❌ Error deleting Firestore data: {error_msg[:200]}")
        firestore_deleted_count = 0

print()
print("=" * 60)
print("✅ Firestore data deletion completed!")
print("=" * 60)

## Section 3: Delete Firestore Database

Delete the Firestore database using gcloud CLI.

**Note:** This will permanently delete the database.

In [ ]:
# Delete the Firestore database using gcloud CLI
print()
print("🗑️  Deleting Firestore database...")
print()
print(f"   Database: {DATABASE_ID}")

result = subprocess.run(
    ["gcloud", "firestore", "databases", "delete", DATABASE_ID,
     f"--project={PROJECT_ID}",
     "--quiet"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"   ✅ Firestore database '{DATABASE_ID}' deleted")
elif "not found" in result.stderr.lower() or "does not exist" in result.stderr.lower():
    print(f"   ⚠️  Database not found (already deleted or never created)")
else:
    print(f"   ⚠️  Could not delete database via CLI: {result.stderr[:200]}")
    print(f"   You can delete it manually at:")
    print(f"   https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")

print()
print("=" * 60)
print("✅ Firestore database cleanup completed!")
print("=" * 60)

## Section 4: Cleanup Summary

In [ ]:
# Summary
print()
print("=" * 70)
print("🎉 CLEANUP COMPLETED!")
print("=" * 70)
print()
print("📊 Summary of deleted resources:")
print()
print("   1. Firestore Data:")
print(f"      - Documents deleted: {firestore_deleted_count if 'firestore_deleted_count' in locals() else 0}")
print(f"      - Collection: {DATASETS_COLLECTION}")
print()
print("   2. Firestore Database:")
print(f"      - Database: {DATABASE_ID}")
print()
print("✅ All resources from setup_firestore.ipynb have been cleaned up!")
print()
print("🔗 Verify in Console:")
print(f"   Firestore Databases: https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")